# Audit de résolution du rang p95

## tl;dr

- L’audit couvre **32 cases POWER_THIRD** et des tailles actives de **6 à 43 signaux**.
- Les **3 désaccords de décision** entre conventions discrètes sont tous dans la référence XTZ/ZEC : 1 lorsque nearest-rank sélectionne le maximum et 2 lorsqu’une seule observation reste au-dessus du rang.
- GRT/MANA conserve **0 dépassement** avec `NEAREST_RANK` et **0 désaccord discret**, malgré 8 cases où le p95 est le maximum.
- Aucune source locale non crypto ne fournit les grains quotidien et 6 h comparables : la réplication inter-univers est non exécutée.
- Le résultat reste `RESEARCH_ONLY` : il valide une échelle de notionnel demandé, ni la liquidité, ni l’alpha, ni une autorisation live.

## Context & Methods

L’estimateur sélectionné reste `NEAREST_RANK`, avec p95 `<= 600 USD` et p95/médiane R7 `<= 2`. Les conventions discrètes `LOWER`, `NEAREST_RANK` et `HIGHER` utilisent exactement les mêmes échantillons.

### Key Assumptions

- La source est l’artefact validé de sensibilité XTZ/ZEC/GRT/MANA, généré sur les folds 2022–2026.
- La classe de résolution dépend uniquement de `n - ceil(0.95 × n)`.
- Les comparaisons sont descriptives ; aucune inférence causale ni re-sélection n’est autorisée.

## Data

### 1. Load and validate the audit artifact

In [1]:
import json
from pathlib import Path

candidate_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(root for root in candidate_roots if (root / 'packages/backtest/package.json').exists())
artifact_path = repo_root / ('packages/backtest/.artifacts/studies/' 'confidence-quantile-sample-size-audit-XTZ-ZEC-GRT-MANA-2022-2026.json')
artifact = json.loads(artifact_path.read_text())
assert artifact['status'] == 'RESEARCH_ONLY'
assert artifact['policy'] == {
    'selectedEstimator': 'NEAREST_RANK',
    'medianEstimator': 'LINEAR_R7',
    'probability': 0.95,
    'maxP95RequestedNotional': 600,
    'maxP95ToMedianRatio': 2,
}
cases = artifact['audit']['cases']
summaries = [row for row in artifact['audit']['summaries'] if row['caseCount'] > 0]
assert len(cases) == 32
print(f"artifact={artifact_path.relative_to(repo_root)}")
print(f"generated_at={artifact['generatedAt']}")
print(f"cases={len(cases)}, n_range={min(x['activeSignalCount'] for x in cases)}..{max(x['activeSignalCount'] for x in cases)}")

artifact=packages/backtest/.artifacts/studies/confidence-quantile-sample-size-audit-XTZ-ZEC-GRT-MANA-2022-2026.json
generated_at=2026-08-19T21:10:48.998Z
cases=32, n_range=6..43


### 2. Reconcile selected-estimator breach counts

In [2]:
for population in ('REFERENCE', 'EXTERNAL'):
    reconciliation = artifact['upstream']['selectedCountReconciliation'][population]
    assert reconciliation['upstream']['absoluteBreachCount'] == reconciliation['recomputed']['absoluteBreachCount']
    assert reconciliation['upstream']['ratioBreachCount'] == reconciliation['recomputed']['ratioBreachCount']
    print(population, reconciliation)

REFERENCE {'upstream': {'absoluteBreachCount': 1, 'ratioBreachCount': 3, 'verdict': 'TAIL_NOT_CONFIRMED'}, 'recomputed': {'absoluteBreachCount': 1, 'ratioBreachCount': 3}}
EXTERNAL {'upstream': {'absoluteBreachCount': 0, 'ratioBreachCount': 0, 'verdict': 'TAIL_CONFIRMED'}, 'recomputed': {'absoluteBreachCount': 0, 'ratioBreachCount': 0}}


## Results

### 3. Resolution classes and decision sensitivity

In [3]:
columns = [
    'populationId', 'resolution', 'caseCount', 'minActiveSignalCount',
    'maxActiveSignalCount', 'selectedAbsoluteBreachCount',
    'selectedRatioBreachCount', 'discreteVerdictDisagreementCount',
]
widths = {column: max(len(column), *(len(str(row[column])) for row in summaries)) for column in columns}
print(' | '.join(column.ljust(widths[column]) for column in columns))
print('-+-'.join('-' * widths[column] for column in columns))
for row in summaries:
    print(' | '.join(str(row[column]).ljust(widths[column]) for column in columns))

populationId | resolution        | caseCount | minActiveSignalCount | maxActiveSignalCount | selectedAbsoluteBreachCount | selectedRatioBreachCount | discreteVerdictDisagreementCount
-------------+-------------------+-----------+----------------------+----------------------+-----------------------------+--------------------------+---------------------------------
REFERENCE    | MAXIMUM           | 8         | 6                    | 16                   | 0                           | 1                        | 1                               
REFERENCE    | ONE_ABOVE         | 8         | 27                   | 39                   | 1                           | 2                        | 2                               
EXTERNAL     | MAXIMUM           | 8         | 8                    | 15                   | 0                           | 0                        | 0                               
EXTERNAL     | ONE_ABOVE         | 6         | 28                   | 36             

### 4. Cases where discrete conventions change the threshold decision

In [4]:
disagreement_rows = []
for case in cases:
    if not case['discreteVerdictDisagreement']:
        continue
    disagreement_rows.append({
        'population': case['populationId'],
        'run': case['runKey'],
        'strategy': case['strategyId'],
        'n': case['activeSignalCount'],
        'resolution': case['resolution'],
        'lower_p95': round(case['p95RequestedNotionalByEstimator']['LOWER'], 2),
        'nearest_p95': round(case['selectedP95RequestedNotional'], 2),
        'nearest_ratio': round(case['selectedP95ToMedianRatio'], 3),
        'spread_usd': round(case['discreteP95SpreadUsd'], 2),
    })
assert len(disagreement_rows) == 3
for row in disagreement_rows:
    print(row)

{'population': 'REFERENCE', 'run': 'ZEC-USD:2022-2023', 'strategy': 'ema-cross', 'n': 16, 'resolution': 'MAXIMUM', 'lower_p95': 200.67, 'nearest_p95': 243.05, 'nearest_ratio': 2.019, 'spread_usd': 42.38}
{'population': 'REFERENCE', 'run': 'ZEC-USD:2022-2023', 'strategy': 'breakout', 'n': 27, 'resolution': 'ONE_ABOVE', 'lower_p95': 391.74, 'nearest_p95': 527.16, 'nearest_ratio': 2.089, 'spread_usd': 135.43}
{'population': 'REFERENCE', 'run': 'XTZ-USD:2024-2025', 'strategy': 'breakout', 'n': 36, 'resolution': 'ONE_ABOVE', 'lower_p95': 500.65, 'nearest_p95': 675.98, 'nearest_ratio': 2.49, 'spread_usd': 175.33}


### 5. Local source availability for a less-correlated universe

In [5]:
source_assessment = artifact['sourceAudit']['assessment']
replication = artifact['crossUniverseReplication']
assert source_assessment['availability'] == 'UNAVAILABLE'
assert replication['status'] == 'NOT_EXECUTED_SOURCE_UNAVAILABLE'
print(source_assessment)
print(replication)

{'availability': 'UNAVAILABLE', 'replicationStatus': 'NOT_EXECUTED_SOURCE_UNAVAILABLE', 'checkedSourceCount': 1, 'eligibleSourceIds': []}
{'status': 'NOT_EXECUTED_SOURCE_UNAVAILABLE', 'executed': False, 'products': None, 'reason': 'no configured non-crypto adapter provides comparable ONE_DAY and SIX_HOUR history in this workspace'}


## Takeaways

- **Le dépassement de référence n’est pas diffus.** Les trois désaccords sont concentrés dans deux classes à faible résolution : `MAXIMUM` et `ONE_ABOVE`.
- **Le petit échantillon n’est pas une cause suffisante.** Les 8 cases externes `MAXIMUM` ne changent jamais de décision selon la convention discrète.
- **La confirmation externe reste bornée avec la règle figée.** GRT/MANA compte zéro dépassement absolu ou relatif avec `NEAREST_RANK`.
- **La réplication inter-univers attend une vraie source comparable.** Aucun proxy crypto supplémentaire ne doit être substitué.
- **Aucune promotion live.** La borne décrit le notionnel demandé ; elle ne mesure pas liquidité, impact de marché ou alpha.